# Lagrangian Trajectory Movies & Truncated PSDs

Load saved trajectory data from notebook 06, generate per-trajectory movies
showing Bz with the particle position and tail, then manually set end times
to exclude unphysical regions and recompute frequency PSDs.

In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from pathlib import Path

from reconn_wave_power.io import read_simulation
from reconn_wave_power.processing import compute_flux_function
from reconn_wave_power.spectrum import compute_psd_time

%matplotlib inline

## Configuration

In [ ]:
INPUT_FILE = "/Users/colby/Research/Projects/My_Projects/2026.Reconn_Wave_Power/test_sim_data/test00/input/input"
OUTPUT_FOLDER = "/Users/colby/Research/Projects/My_Projects/2026.Reconn_Wave_Power/test_sim_data/test00/Output"
BATCH_FILE = "lagrangian_batch.nc"

MOVIE_DIR = os.path.join(os.path.dirname(os.getcwd()), "movies") if os.path.basename(os.getcwd()) == "notebooks" else "movies"
os.makedirs(MOVIE_DIR, exist_ok=True)

FRAME_STRIDE = 10       # render every Nth simulation frame (reduces 1001 → ~100)
TAIL_LEN = 20           # number of previous *strided* frames shown as tail
MOVIE_FPS = 20
MOVIE_DPI = 150
PSD_METHOD = "fft"
N_CONTOURS = 20         # number of flux-function contour levels (field lines)

## Load Data

In [ ]:
# Saved trajectory + sampled field data
ds_batch = xr.open_dataset(BATCH_FILE)
print(ds_batch)

x_traj = ds_batch["x_traj"].values        # (N_total, n_times)
y_traj = ds_batch["y_traj"].values
active_mask = ds_batch["active_mask"].values.astype(bool)
t_traj = ds_batch["time"].values
x0s = ds_batch["x0"].values
y0s = ds_batch["y0"].values
N_total = len(x0s)
dt = ds_batch.attrs["dt"]
n_lines = ds_batch.attrs["n_lines"]
n_per_line = ds_batch.attrs["n_per_line"]

print(f"\nTrajectories: {N_total}, Timesteps: {len(t_traj)}, dt: {dt}")

In [ ]:
# Load simulation Bz for movie frames
ds = read_simulation(
    input_file=INPUT_FILE,
    output_folder=OUTPUT_FOLDER,
    fields=("B",),
    progress=True,
)
print(ds)

## Compute Global Color Limits

In [ ]:
#bz_min = float(ds["Bz"].min())
#bz_max = float(ds["Bz"].max())
bz_min = 0
bz_max = .5
vlim = max(abs(bz_min), abs(bz_max))
print(f"Bz range: [{bz_min:.4f}, {bz_max:.4f}], symmetric clim: +/-{vlim:.4f}")

## Preload Bz Frames

Batch-load strided Bz frames into memory so each timestep is read exactly once.

In [ ]:
import dask

nt = len(t_traj)
frame_indices = np.arange(0, nt, FRAME_STRIDE)
n_frames = len(frame_indices)
print(f"Strided frames: {n_frames} (stride={FRAME_STRIDE}, original={nt})")

# Batch-load all strided Bz, Bx, and By frames via dask.compute in chunks of 64
BATCH_SIZE = 64
bz_frames = np.empty((n_frames, len(ds.x), len(ds.y)), dtype=np.float32)
bx_frames = np.empty((n_frames, len(ds.x), len(ds.y)), dtype=np.float32)
by_frames = np.empty((n_frames, len(ds.x), len(ds.y)), dtype=np.float32)

for batch_start in range(0, n_frames, BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, n_frames)
    delayed_bz = [ds["Bz"].isel(time=int(frame_indices[i])).data
                  for i in range(batch_start, batch_end)]
    delayed_bx = [ds["Bx"].isel(time=int(frame_indices[i])).data
                  for i in range(batch_start, batch_end)]
    delayed_by = [ds["By"].isel(time=int(frame_indices[i])).data
                  for i in range(batch_start, batch_end)]
    results = dask.compute(*(delayed_bz + delayed_bx + delayed_by))
    n_batch = batch_end - batch_start
    for j in range(n_batch):
        bz_frames[batch_start + j] = results[j]
        bx_frames[batch_start + j] = results[n_batch + j]
        by_frames[batch_start + j] = results[2 * n_batch + j]
    print(f"  Loaded frames {batch_start}–{batch_end - 1} / {n_frames - 1}")

print(f"bz_frames shape: {bz_frames.shape}, size: {bz_frames.nbytes / 1e6:.1f} MB")
print(f"bx_frames shape: {bx_frames.shape}, size: {bx_frames.nbytes / 1e6:.1f} MB")
print(f"by_frames shape: {by_frames.shape}, size: {by_frames.nbytes / 1e6:.1f} MB")

# Compute flux function ψ using two-step method (Bx and By)
x_coords = ds.x.values
y_coords = ds.y.values
dx = x_coords[1] - x_coords[0]
dy = y_coords[1] - y_coords[0]
psi_frames = np.zeros_like(bx_frames)
# Step 1: boundary at x=0 — integrate Bx along y
psi_frames[:, 0, 1:] = np.cumsum(bx_frames[:, 0, 1:], axis=-1) * dy
# Step 2: interior — extend using -∂ψ/∂x = By
psi_frames[:, 1:, :] = psi_frames[:, 0:1, :] - np.cumsum(by_frames[:, 1:, :], axis=-2) * dx
print(f"psi_frames shape: {psi_frames.shape}, size: {psi_frames.nbytes / 1e6:.1f} MB")

## Generate Movies

In [ ]:
if 0:
    for traj_idx in range(N_total):
        x0 = x0s[traj_idx]
        y0 = y0s[traj_idx]
        fname = os.path.join(
            MOVIE_DIR,
            f"trajectory_{traj_idx:02d}_x0_{x0:.1f}_y0_{y0:.1f}.mp4",
        )
    
        fig, ax = plt.subplots(figsize=(12, 4))
        im = ax.pcolormesh(
            ds.x, ds.y, bz_frames[0].T, shading="auto", cmap="RdBu_r",
            vmin=-vlim, vmax=vlim,
        )
        # Initial flux-function contours (field lines)
        contour_set = ax.contour(
            ds.x.values, ds.y.values, psi_frames[0].T,
            levels=N_CONTOURS, colors="k", linewidths=0.5,
        )
        (tail_line,) = ax.plot([], [], "-", color="black", lw=1.5, alpha=0.7)
        (marker,) = ax.plot([], [], "o", color="black", ms=6, mec="white", mew=0.8)
        ax.set_xlabel(r"x [$d_i$]")
        ax.set_ylabel(r"y [$d_i$]")
        ax.set_aspect("equal")
        plt.colorbar(im, ax=ax, label="Bz")
        title = ax.set_title("")
    
        xt = x_traj[traj_idx]
        yt = y_traj[traj_idx]
    
        def update(frame_idx, xt=xt, yt=yt):
            nonlocal contour_set
            im.set_array(bz_frames[frame_idx].T.ravel())
            # Update flux-function contours: remove old, draw new
            for c in contour_set.collections:
                c.remove()
            contour_set = ax.contour(
                ds.x.values, ds.y.values, psi_frames[frame_idx].T,
                levels=N_CONTOURS, colors="k", linewidths=0.5,
            )
            # Map strided frame index back to original time index for trajectory coords
            orig_idx = frame_indices[frame_idx]
            # Tail: look back TAIL_LEN strided frames
            t_start = max(0, frame_idx - TAIL_LEN)
            tail_orig = frame_indices[t_start:frame_idx + 1]
            tail_line.set_data(xt[tail_orig], yt[tail_orig])
            # Current position
            marker.set_data([xt[orig_idx]], [yt[orig_idx]])
            title.set_text(
                f"Trajectory {traj_idx:02d}  "
                f"(x0={x0:.1f}, y0={y0:.1f})  "
                f"t = {t_traj[orig_idx]:.1f} $\\Omega_{{ci}}^{{-1}}$"
            )
            return im, tail_line, marker, title
    
        anim = FuncAnimation(fig, update, frames=n_frames, interval=50, blit=False)
        plt.close(fig)
        anim.save(fname, writer="ffmpeg", fps=MOVIE_FPS, dpi=MOVIE_DPI)
        print(f"[{traj_idx + 1}/{N_total}] Saved: {fname}")
    
    print("\nAll movies saved.")

In [ ]:
print(len(t_traj))

## Define Per-Trajectory End Times

After reviewing the movies, set the end time index for each trajectory.
Trajectories that stay physical for the full run keep the default (last index).
Edit entries below for trajectories that enter unphysical regions.

In [ ]:
# Default: use all timesteps
end_time_idx = np.full(N_total, len(t_traj), dtype=int)

# --- Manually override unphysical trajectories ---
end_times = []
# 1/4 Line --   0     1     2    3     4     5     6     7    8    9 
end_times += [880, 1020, 1120, 820, 1100, 1000, 1000, 1060, 900, 800] 
# 1/4 Line --  10   11   12   13   14   15   16   17   18   19 
end_times += [840, 700, 600, 500, 500, 500, 500, 500, 500, 720]

# 1/2 Line --   20   21   22   23   24   25   26   27   28   29 
end_times += [1000, 900, 720, 520, 500, 500, 500, 500, 660, 920]
# 1/2 Line --  30   31   32   33   34   35   36   37   38   39 
end_times += [1000, 900, 720, 520, 500, 500, 500, 500, 740, 920]

# 3/4 Line --   40   41   42   43   44   45   46   47   48   49 
end_times += [900, 740, 540, 500, 500, 500, 500, 500, 500, 780]
# 3/4 Line --  50   51   52   53   54   55   56   57   58   59 
end_times += [960, 1040, 1060, 640, 500, 500, 500, 500, 1020, 1020]

end_time_idx = np.array(end_times)//2 + 1

""" Notes:
#4-6 are lower left in the island
#7 and 8 are good ones
#13 - 18 are just going straight into the x-line
#23 - 27 in the current sheet
"""

# end_time_idx[3] = 500
# end_time_idx[17] = 750

# Summary table
print(f"{'Traj':>4s}  {'x0':>8s}  {'y0':>8s}  {'end_idx':>7s}  {'end_time':>10s}")
print("-" * 45)
for i in range(N_total):
    t_end = t_traj[end_time_idx[i] - 1] if end_time_idx[i] > 0 else 0.0
    flag = "  <-- truncated" if end_time_idx[i] < len(t_traj) else ""
    print(f"{i:4d}  {x0s[i]:8.1f}  {y0s[i]:8.1f}  {end_time_idx[i]:7d}  {t_end:10.1f}{flag}")

## Compute Truncated PSDs

In [ ]:
COMPONENTS = ["Ex", "Ey", "Ez", "Bx", "By", "Bz"]
DERIVED = ["Sx", "Sy", "Sz", "uE", "uB"]
psd_results = {}   # comp -> list of arrays (varying lengths)
freq_results = {}  # comp -> list of frequency arrays

for comp in COMPONENTS:
    print(f"Computing PSDs for {comp}...")
    sampled = ds_batch[f"{comp}_sampled"].values  # (N_total, n_times)
    psd_list = []
    freq_list = []
    for i in range(N_total):
        end = end_time_idx[i]
        vals = sampled[i, :end]
        t_valid = t_traj[:end]
        _dt = t_traj[2] - t_traj[1]
        da = xr.DataArray(vals, dims=("time",), coords={"time": t_valid})
        f, Pxx = compute_psd_time(da, dt=_dt, method=PSD_METHOD)
        freq_list.append(f)
        psd_list.append(Pxx)
    psd_results[comp] = psd_list
    freq_results[comp] = freq_list

# --- Derived quantities: Poynting vector and energy densities ---
Ex_s = ds_batch["Ex_sampled"].values
Ey_s = ds_batch["Ey_sampled"].values
Ez_s = ds_batch["Ez_sampled"].values
Bx_s = ds_batch["Bx_sampled"].values
By_s = ds_batch["By_sampled"].values
Bz_s = ds_batch["Bz_sampled"].values

derived_fields = {
    "Sx": Ey_s * Bz_s - Ez_s * By_s,
    "Sy": Ez_s * Bx_s - Ex_s * Bz_s,
    "Sz": Ex_s * By_s - Ey_s * Bx_s,
    "uE": 0.5 * (Ex_s**2 + Ey_s**2 + Ez_s**2),
    "uB": 0.5 * (Bx_s**2 + By_s**2 + Bz_s**2),
}

for comp in DERIVED:
    print(f"Computing PSDs for {comp}...")
    sampled = derived_fields[comp]
    psd_list = []
    freq_list = []
    for i in range(N_total):
        end = end_time_idx[i]
        vals = sampled[i, :end]
        t_valid = t_traj[:end]
        _dt = t_traj[2] - t_traj[1]
        da = xr.DataArray(vals, dims=("time",), coords={"time": t_valid})
        f, Pxx = compute_psd_time(da, dt=_dt, method=PSD_METHOD)
        freq_list.append(f)
        psd_list.append(Pxx)
    psd_results[comp] = psd_list
    freq_results[comp] = freq_list

print(f"\nPSD lengths per trajectory: {[len(p) for p in psd_results[COMPONENTS[0]]]}")

## Plot PSDs

In [ ]:
FIELD = "Bz"

# Unique x-line starting positions
x_lines = np.unique(x0s)
colors = plt.cm.tab10(np.linspace(0, 1, len(x_lines)))

fig, ax = plt.subplots(figsize=(10, 6))
for i_line, xl in enumerate(x_lines):
    mask = x0s == xl
    idxs = np.where(mask)[0]
    for j in idxs:
        ax.semilogy(freq_results[FIELD][j], psd_results[FIELD][j],
                    color=colors[i_line], alpha=0.5, lw=0.8)
    ax.semilogy([], [], color=colors[i_line], lw=2, label=f"x0 = {xl:.1f}")

ax.set_xlabel(r"frequency [$\Omega_{ci}$]")
ax.set_ylabel("PSD")
ax.set_title(f"Truncated Lagrangian PSD of {FIELD}")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
_k = 'Bx'

compute_psd_time??
print('dt', dt)
_dt = t_traj[3] - t_traj[2]
print(_dt)
print(np.pi/_dt)
_j = 0

for _l in range(60):
    cid = _l/60.
    cid = plt.cm.jet(cid)
    _f = freq_results[_k][_l]
    _Nt = len(_f)*2 # The two is because the array is halfed for rfftfreq
    print(len(_f), 1./(_Nt*_dt), _f[1]) # The second 2 is for the 

    _f = freq_results[_k][_l]
    _p = psd_results[_k][_l]
    plt.loglog(_f, _p, color=cid)

    # Lets make sure we have the right frequency
    # 

In [ ]:
_Nt = 100
print(_dt)
print(np.fft.rfftfreq(_Nt, _dt))
print(1./_Nt/2)

## Save Truncated Results

In [ ]:
# Pad ragged PSDs with NaN to max length for NetCDF storage
ALL_COMPS = COMPONENTS + DERIVED
max_nfreq = max(len(p) for p in psd_results[COMPONENTS[0]])
freq_max = freq_results[COMPONENTS[0]][np.argmax([len(f) for f in freq_results[COMPONENTS[0]]])]

def pad_to_length(arr, length):
    padded = np.full(length, np.nan)
    padded[:len(arr)] = arr
    return padded

ds_out = xr.Dataset(
    {
        "end_time_idx": (["trajectory"], end_time_idx),
        "x0": (["trajectory"], x0s),
        "y0": (["trajectory"], y0s),
        "n_freq": (["trajectory"], np.array([len(p) for p in psd_results[COMPONENTS[0]]])),
    },
    coords={
        "trajectory": np.arange(N_total),
        "frequency": freq_max,
    },
    attrs={
        "source_file": BATCH_FILE,
        "psd_method": PSD_METHOD,
        "dt": dt,
    },
)

for comp in ALL_COMPS:
    padded = np.array([pad_to_length(p, max_nfreq) for p in psd_results[comp]])
    ds_out[f"psd_{comp}"] = (["trajectory", "frequency"], padded)

out_path = Path(".") / "lagrangian_batch_truncated.nc"
ds_out.to_netcdf(out_path)
print(f"Saved to {out_path}")
print(ds_out)

## Per-Trajectory PSD Plots

Save an individual PSD figure for each trajectory (all 6 field components)
so it can be compared side-by-side with the trajectory movie.

In [ ]:
PSD_DIR = os.path.join(os.path.dirname(os.getcwd()), "psd_plots") if os.path.basename(os.getcwd()) == "notebooks" else "psd_plots"
os.makedirs(PSD_DIR, exist_ok=True)

E_COMPS = ["Ex", "Ey", "Ez"]
B_COMPS = ["Bx", "By", "Bz"]
S_COMPS = ["Sx", "Sy", "Sz"]
ENERGY_COMPS = ["uE", "uB"]
COMP_COLORS = {"Ex": "C0", "Ey": "C1", "Ez": "C2",
               "Bx": "C3", "By": "C4", "Bz": "C5",
               "Sx": "C0", "Sy": "C1", "Sz": "C2",
               "uE": "C6", "uB": "C7"}

# --- Compute global axis limits across all trajectories ---
all_freqs_min, all_freqs_max = np.inf, -np.inf
e_psd_min, e_psd_max = np.inf, -np.inf
b_psd_min, b_psd_max = np.inf, -np.inf
d_psd_min, d_psd_max = np.inf, -np.inf  # derived quantities

for traj_idx in range(N_total):
    for comp in COMPONENTS + DERIVED:
        f = freq_results[comp][traj_idx]
        p = psd_results[comp][traj_idx]
        f_pos, p_pos = f[1:], p[1:]  # skip DC
        if len(f_pos) == 0:
            continue
        all_freqs_min = min(all_freqs_min, f_pos[0])
        all_freqs_max = max(all_freqs_max, f_pos[-1])
        p_valid = p_pos[p_pos > 0]
        if len(p_valid) == 0:
            continue
        if comp in E_COMPS:
            e_psd_min = min(e_psd_min, p_valid.min())
            e_psd_max = max(e_psd_max, p_valid.max())
        elif comp in B_COMPS:
            b_psd_min = min(b_psd_min, p_valid.min())
            b_psd_max = max(b_psd_max, p_valid.max())
        else:
            d_psd_min = min(d_psd_min, p_valid.min())
            d_psd_max = max(d_psd_max, p_valid.max())

# Add some padding in log space (half a decade)
FREQ_LIM = (all_freqs_min * 0.8, all_freqs_max * 1.2)
E_PSD_LIM = (e_psd_min * 0.3, e_psd_max * 3.0)
B_PSD_LIM = (b_psd_min * 0.3, b_psd_max * 3.0)
D_PSD_LIM = (d_psd_min * 0.3, d_psd_max * 3.0)

print(f"Freq limits:     {FREQ_LIM}")
print(f"E PSD limits:    {E_PSD_LIM}")
print(f"B PSD limits:    {B_PSD_LIM}")
print(f"Derived limits:  {D_PSD_LIM}")

# --- Generate per-trajectory plots with fixed axes ---
for traj_idx in range(N_total):
    x0 = x0s[traj_idx]
    y0 = y0s[traj_idx]
    end = end_time_idx[traj_idx]
    t_end = t_traj[end - 1] if end > 0 else 0.0

    fig, axes = plt.subplots(1, 3, figsize=(20, 5))

    # Left panel: E-field components
    ax = axes[0]
    for comp in E_COMPS:
        f = freq_results[comp][traj_idx]
        p = psd_results[comp][traj_idx]
        ax.loglog(f[1:], p[1:], color=COMP_COLORS[comp], lw=1.2, label=comp)
    ax.set_xlim(FREQ_LIM)
    ax.set_ylim(E_PSD_LIM)
    ax.set_xlabel(r"frequency [$\Omega_{ci}$]")
    ax.set_ylabel("PSD")
    ax.set_title("E-field")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, which="both")

    # Middle panel: B-field components
    ax = axes[1]
    for comp in B_COMPS:
        f = freq_results[comp][traj_idx]
        p = psd_results[comp][traj_idx]
        ax.loglog(f[1:], p[1:], color=COMP_COLORS[comp], lw=1.2, label=comp)
    ax.set_xlim(FREQ_LIM)
    ax.set_ylim(B_PSD_LIM)
    ax.set_xlabel(r"frequency [$\Omega_{ci}$]")
    ax.set_ylabel("PSD")
    ax.set_title("B-field")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, which="both")

    # Right panel: Poynting vector + energy densities
    ax = axes[2]
    for comp in S_COMPS:
        f = freq_results[comp][traj_idx]
        p = psd_results[comp][traj_idx]
        ax.loglog(f[1:], p[1:], color=COMP_COLORS[comp], lw=1.2, label=comp)
    for comp in ENERGY_COMPS:
        f = freq_results[comp][traj_idx]
        p = psd_results[comp][traj_idx]
        ax.loglog(f[1:], p[1:], color=COMP_COLORS[comp], lw=1.2,
                  linestyle="--", label=comp)
    ax.set_xlim(FREQ_LIM)
    ax.set_ylim(D_PSD_LIM)
    ax.set_xlabel(r"frequency [$\Omega_{ci}$]")
    ax.set_ylabel("PSD")
    ax.set_title("Poynting & Energy")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, which="both")

    fig.suptitle(
        f"Trajectory {traj_idx:02d}  (x0={x0:.1f}, y0={y0:.1f})  "
        f"t_end={t_end:.0f} $\\Omega_{{ci}}^{{-1}}$ ({end} steps)",
        fontsize=12,
    )
    plt.tight_layout()

    fname = os.path.join(
        PSD_DIR,
        f"psd_{traj_idx:02d}_x0_{x0:.1f}_y0_{y0:.1f}.png",
    )
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    plt.close(fig)

print(f"Saved {N_total} PSD plots to {PSD_DIR}/")